# MILP v2 — Campus Microgrid Sizing (Gurobi, Optimized)

Same two-stage stochastic MILP as v1, with Gurobi-specific optimizations:

- **Vectorized model construction**: Pre-computed numpy parameter arrays + `addConstrs` generators + `quicksum`
- **BESS charge/discharge exclusivity**: Binary variables prevent simultaneous charge & discharge (true MILP)
- **Solver tuning**: Barrier method, aggressive presolve & cuts, MIP gap control

Requires Gurobi academic license. Uses shared config/data from `milp_common.py`.

In [ ]:
import numpy as np
import gurobipy as gp
from gurobipy import GRB
import time
from milp_common import get_config, load_data, print_results

In [ ]:
CFG = get_config()
scenarios, meta, repday_data, n_repdays, n_scenarios, n_hours = load_data(CFG)

## Pre-compute Parameter Arrays

Flatten all per-(d,s,t) parameters into 1-D numpy arrays of length N = 11,400.
Linear index: `i = d * (S * T) + s * T + t`

In [ ]:
N = n_repdays * n_scenarios * n_hours
PV_RATING = 50.0

pv_coeff = np.empty(N)
load_arr = np.empty(N)
pw_arr   = np.empty(N)
tou_arr  = np.empty(N)

for d in range(n_repdays):
    dd = repday_data[d]
    w = dd['weight']
    tou = dd['tou']
    for s in range(n_scenarios):
        sc = dd['scenarios'][s]
        start = d * (n_scenarios * n_hours) + s * n_hours
        sl = slice(start, start + n_hours)
        pv_coeff[sl] = sc['pv_kw'] / PV_RATING
        load_arr[sl] = sc['load_kw']
        pw_arr[sl]   = sc['prob'] * w
        tou_arr[sl]  = tou

load_total_val = float(np.sum(pw_arr * load_arr))

# SOC chaining: identify first-hour indices and previous-hour mapping
is_first_hour = np.zeros(N, dtype=bool)
for d in range(n_repdays):
    for s in range(n_scenarios):
        is_first_hour[d * (n_scenarios * n_hours) + s * n_hours] = True

first_hours = np.where(is_first_hour)[0]
later_hours = np.where(~is_first_hour)[0]
prev_idx = np.arange(N) - 1  # valid only for later_hours

print(f'N = {N:,} decision points')
print(f'First hours: {len(first_hours)}, Later hours: {len(later_hours)}')

## Build Model (Vectorized)

In [ ]:
t_build = time.time()
m = gp.Model('MicrogridSizing_v2')

# === Stage 1: Capacity variables ===
cap_pv       = m.addVar(ub=CFG['pv_max_kw'], name='cap_pv')
cap_bess_p   = m.addVar(ub=CFG['bess_p_max_kw'], name='cap_bess_p')
cap_bess_e   = m.addVar(ub=CFG['bess_e_max_kwh'], name='cap_bess_e')
cap_contract = m.addVar(name='cap_contract')

# === Stage 2: Dispatch variables (MVar arrays) ===
p_grid = m.addMVar(N, name='grid')
p_curt = m.addMVar(N, name='curt')
p_export = m.addMVar(N, name='exp')
p_ch  = m.addMVar(N, name='ch')
p_dis = m.addMVar(N, name='dis')
soc   = m.addMVar(N, name='soc')
over_contract = m.addMVar(N, name='oc')

# === Binary BESS exclusivity (NEW in v2) ===
bin_ch  = m.addMVar(N, vtype=GRB.BINARY, name='bin_ch')
bin_dis = m.addMVar(N, vtype=GRB.BINARY, name='bin_dis')

m.update()
print(f'Variables: {m.NumVars:,} ({2*N:,} binary)')

## Constraints (Batched via `addConstrs`)

In [ ]:
eff_ch = CFG['eff_charge']
eff_dis_inv = 1.0 / CFG['eff_discharge']
soc_init_frac = CFG['soc_init']
soc_min_frac = CFG['soc_min']
soc_max_frac = CFG['soc_max']

# --- Power balance ---
m.addConstrs(
    (pv_coeff[i] * cap_pv + p_grid[i] + p_dis[i]
     == load_arr[i] + p_ch[i] + p_curt[i] + p_export[i]
     for i in range(N)),
    name='bal'
)

# --- BESS exclusivity: can't charge and discharge simultaneously ---
m.addConstrs(
    (bin_ch[i] + bin_dis[i] <= 1 for i in range(N)),
    name='excl'
)
m.addConstrs(
    (p_ch[i] <= cap_bess_p * bin_ch[i] for i in range(N)),
    name='ch_lim'
)
m.addConstrs(
    (p_dis[i] <= cap_bess_p * bin_dis[i] for i in range(N)),
    name='dis_lim'
)

# --- SOC dynamics: first hour (soc_prev = soc_init * cap_bess_e) ---
m.addConstrs(
    (soc[i] == soc_init_frac * cap_bess_e
              + eff_ch * p_ch[i] - eff_dis_inv * p_dis[i]
     for i in first_hours),
    name='soc_init'
)

# --- SOC dynamics: subsequent hours ---
m.addConstrs(
    (soc[i] == soc[prev_idx[i]]
              + eff_ch * p_ch[i] - eff_dis_inv * p_dis[i]
     for i in later_hours),
    name='soc_cont'
)

# --- SOC bounds ---
m.addConstrs(
    (soc[i] >= soc_min_frac * cap_bess_e for i in range(N)),
    name='soc_lo'
)
m.addConstrs(
    (soc[i] <= soc_max_frac * cap_bess_e for i in range(N)),
    name='soc_hi'
)

# --- Over-contract penalty ---
m.addConstrs(
    (over_contract[i] >= p_grid[i] - cap_contract for i in range(N)),
    name='oc'
)

# --- Export limit ---
m.addConstrs(
    (p_export[i] <= pv_coeff[i] * cap_pv for i in range(N)),
    name='exp_lim'
)

m.update()
print(f'Constraints: {m.NumConstrs:,}')

## Objective (via `quicksum`)

In [ ]:
# Investment cost
inv_cost = (
    cap_pv * CFG['capex_pv_per_kw'] * CFG['crf_pv']
    + cap_bess_p * CFG['capex_bess_power_per_kw'] * CFG['crf_bess']
    + cap_bess_e * (CFG['capex_bess_energy_per_kwh'] * CFG['crf_bess']
                    + CFG['capex_bess_energy_per_kwh'] * CFG['fom_bess_rate'])
    + cap_contract * CFG['contract_price_per_kw_month'] * 12
)

# Operating cost via quicksum (avoids repeated LinExpr reallocation)
penalty = CFG['penalty_per_kw']
fit = CFG['feed_in_tariff']
deg = CFG['bess_degradation_cost']

op_cost = gp.quicksum(
    pw_arr[i] * (
        p_grid[i] * float(tou_arr[i])
        + over_contract[i] * penalty
        - p_export[i] * fit
        + (p_ch[i] + p_dis[i]) * deg
    )
    for i in range(N)
)

m.setObjective(inv_cost + op_cost, GRB.MINIMIZE)

# RE target constraint
re_expr = gp.quicksum(
    pw_arr[i] * (load_arr[i] - p_grid[i])
    for i in range(N)
)
m.addConstr(re_expr >= CFG['re_target'] * load_total_val, name='RE30')

m.update()
build_time = time.time() - t_build
print(f'Model built in {build_time:.1f}s')
print(f'Total constraints: {m.NumConstrs:,}')
print(f'Total variables:   {m.NumVars:,} ({2*N:,} binary)')

## Solve (Tuned Parameters)

In [ ]:
m.Params.TimeLimit = CFG['time_limit']
m.Params.Threads   = 0      # auto-detect all cores
m.Params.MIPGap    = 1e-4   # 0.01% optimality gap
m.Params.Presolve  = 2      # aggressive presolve
m.Params.Method    = 2      # barrier for root LP relaxation
m.Params.MIPFocus  = 1      # prioritize feasible solutions
m.Params.Cuts      = 2      # aggressive cuts

t0 = time.time()
m.optimize()
solve_time = time.time() - t0

print(f'\nStatus: {m.Status} ({"Optimal" if m.Status == GRB.OPTIMAL else "See log"})')
if m.Status in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
    print(f'Objective: {m.ObjVal:,.0f} TWD')
    print(f'MIP Gap: {m.MIPGap:.6f}')
print(f'Solve time: {solve_time:.1f}s')

## Results

In [ ]:
if m.Status in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
    capacities = (cap_pv.X, cap_bess_p.X, cap_bess_e.X, cap_contract.X)
    obj_val = m.ObjVal

    # Vectorized RE extraction
    grid_vals = np.array([p_grid[i].X for i in range(N)])
    re_val = float(np.sum(pw_arr * (load_arr - grid_vals)))
    load_val = load_total_val

    results = print_results(
        capacities, obj_val, CFG, re_val, load_val, solve_time,
        repday_data, n_repdays, n_scenarios, n_hours, scenarios
    )

    # Check exclusivity: how many hours have simultaneous charge & discharge?
    ch_vals = np.array([bin_ch[i].X for i in range(N)])
    dis_vals = np.array([bin_dis[i].X for i in range(N)])
    simultaneous = np.sum((ch_vals > 0.5) & (dis_vals > 0.5))
    print(f'\nBESS exclusivity check: {simultaneous} hours with simultaneous charge/discharge (should be 0)')
else:
    print(f'Model not solved to optimality. Status: {m.Status}')